# 03 - E-distance 示例

读取 AnnData → 过滤 control 和 GeneA → 标准归一化 → log1p → PCA → 计算 E-distance → 运行 E-test → 解释距离、p 值和校正后 p 值。

优先使用维护活跃的 pertpy；如同时使用 scperturb 包，记录二者版本及结果差异。

In [ ]:
import sys
sys.path.insert(0, '..')
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd
from pathlib import Path

# 加载最小示例或真实数据
use_minimal = True
if use_minimal:
    adata = ad.read_h5ad('../examples/minimal_perturbation.h5ad')
    print(f'Using minimal example: {adata.shape}')
else:
    adata = ad.read_h5ad('../data/raw/AdamsonWeissman2016_GSM2406675_10X001.h5ad')
    print(f'Using Adamson data: {adata.shape}')

print(f'Perturbations: {dict(adata.obs["perturbation"].value_counts())}')

In [ ]:
# 过滤 control 和 GeneA
adata_sub = adata[adata.obs['perturbation'].isin(['control', 'GeneA'])].copy()
print(f'Subset: {adata_sub.shape}')
print(f'Groups: {dict(adata_sub.obs["perturbation"].value_counts())}')

In [ ]:
# 标准归一化 → log1p → PCA
sc.pp.normalize_total(adata_sub, target_sum=1e4)
sc.pp.log1p(adata_sub)
sc.pp.pca(adata_sub, n_comps=min(2, adata_sub.n_obs - 1, adata_sub.n_vars - 1))
print(f'PCA shape: {adata_sub.obsm["X_pca"].shape}')
print(adata_sub.obsm['X_pca'])

In [ ]:
# 计算 E-distance (使用 pertpy)
try:
    import pertpy as pt
    print(f'pertpy version: {pt.__version__}')
    
    ed = pt.tl.EnergyDistance()
    # 计算 control vs GeneA 的 E-distance
    results = ed.compute(
        adata_sub,
        groupby='perturbation',
        contrast='control',
        target='GeneA',
    )
    print('E-distance results:')
    print(results)
    
except ImportError:
    print('pertpy not installed. Using manual E-distance calculation.')
    
    # 手动计算 E-distance (基于 PCA 空间中的欧氏距离)
    from scipy.spatial.distance import cdist
    
    pca = adata_sub.obsm['X_pca']
    ctrl_mask = adata_sub.obs['perturbation'] == 'control'
    genea_mask = adata_sub.obs['perturbation'] == 'GeneA'
    
    ctrl_pca = pca[ctrl_mask]
    genea_pca = pca[genea_mask]
    
    # E-distance = 2*E|X-Y| - E|X-X'| - E|Y-Y'|
    d_xy = cdist(ctrl_pca, genea_pca).mean()
    d_xx = cdist(ctrl_pca, ctrl_pca).mean()
    d_yy = cdist(genea_pca, genea_pca).mean()
    e_distance = 2 * d_xy - d_xx - d_yy
    
    print(f'E-distance (control vs GeneA): {e_distance:.4f}')
    print(f'  E|X-Y|: {d_xy:.4f}')
    print(f'  E|X-X\'|: {d_xx:.4f}')
    print(f'  E|Y-Y\'|: {d_yy:.4f}')

In [ ]:
# E-test (置换检验)
from scipy.spatial.distance import cdist

pca = adata_sub.obsm['X_pca']
ctrl_mask = (adata_sub.obs['perturbation'] == 'control').values
genea_mask = (adata_sub.obs['perturbation'] == 'GeneA').values

def compute_ed(pca, mask1, mask2):
    d_xy = cdist(pca[mask1], pca[mask2]).mean()
    d_xx = cdist(pca[mask1], pca[mask1]).mean()
    d_yy = cdist(pca[mask2], pca[mask2]).mean()
    return 2 * d_xy - d_xx - d_yy

observed_ed = compute_ed(pca, ctrl_mask, genea_mask)

# 置换检验
n_permutations = 1000
null_dist = []
n_total = len(ctrl_mask) + len(genea_mask)
combined = np.arange(n_total)

np.random.seed(42)
for _ in range(n_permutations):
    perm = np.random.permutation(combined)
    mask1_perm = perm[:ctrl_mask.sum()]
    mask2_perm = perm[ctrl_mask.sum():]
    null_ed = compute_ed(pca, mask1_perm, mask2_perm)
    null_dist.append(null_ed)

p_value = (np.array(null_dist) >= observed_ed).mean()
print(f'Observed E-distance: {observed_ed:.4f}')
print(f'p-value (permutation test): {p_value:.4f}')
print(f'Null distribution: mean={np.mean(null_dist):.4f}, std={np.std(null_dist):.4f}')

## 结果解释

- **E-distance**: 衡量两个细胞分布之间的统计距离。值越大，表示扰动效应越强。
- **p 值**: 置换检验的 p 值，表示在零假设下观察到当前 E-distance 或更大值的概率。
- **校正后 p 值**: 多重检验校正（如 Bonferroni 或 Benjamini-Hochberg），用于控制假阳性率。

注意：最小示例数据量太小（每组仅 2 个细胞），结果仅供流程测试，不可用于复现论文指标。